Práctica HW2 Big Data. \
Marcel, Pau, Ane

# **Imports**

In [ ]:
!pip install pyspark==4.1.1


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from pyspark.sql.functions import col, sum, year, round, hour, dayofweek, month, when
from pyspark.sql.functions import count, rank, desc, avg, dense_rank, sum, round
from pyspark.sql.window import Window


# **Introducing the Data: Yellow Taxis from January and February 2026**

In [ ]:
import os
import shutil
import subprocess

# Aseguramos que Java esta disponible y que javahome esta configurado
java_cmd = shutil.which("java")

# Si no encuentra java o si la variable de entorno no existe
if not java_cmd or not os.environ.get("JAVA_HOME"):
    try:
        # Actualizamos o instalamos java
        subprocess.run(["bash", "-lc", "apt-get update -y && apt-get install -y openjdk-11-jdk-headless"], check=True)
        # Ruta
        java_home_guess = "/usr/lib/jvm/java-11-openjdk-amd64"
        if os.path.isdir(java_home_guess):
            os.environ["JAVA_HOME"] = java_home_guess
            os.environ["PATH"] = f"{java_home_guess}/bin:" + os.environ.get("PATH", "")
    except Exception:
        # Si la instalacion de java falla, intentamos la version 17
        try:
            subprocess.run(["bash", "-lc", "apt-get update -y && apt-get install -y openjdk-17-jdk-headless"], check=True)
            # Ruta
            java_home_guess = "/usr/lib/jvm/java-17-openjdk-amd64"
            if os.path.isdir(java_home_guess):
                os.environ["JAVA_HOME"] = java_home_guess
                os.environ["PATH"] = f"{java_home_guess}/bin:" + os.environ.get("PATH", "")
        except Exception:
            # Silenciamos el error si todo falla
            pass

# Creamos la sesion de Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName('NYC_Taxi_Analysis') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

# Imprimir resumen de la sesion de Spark (para ver si esta funcionando bien)
spark

Hit:1 http://deb.debian.org/debian bookworm InRelease
Hit:2 http://deb.debian.org/debian bookworm-updates InRelease
Hit:3 http://deb.debian.org/debian-security bookworm-security InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
Package openjdk-11-jdk-headless is not available, but is referred to by another package.
This may mean that the package is missing, has been obsoleted, or
is only available from another source
However the following packages replace it:
  openjdk-17-jre-headless

E: Package 'openjdk-11-jdk-headless' has no installation candidate
Hit:1 http://deb.debian.org/debian bookworm InRelease
Hit:2 http://deb.debian.org/debian bookworm-updates InRelease
Hit:3 http://deb.debian.org/debian-security bookworm-security InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
openjdk-17-jdk-headless is already the newest version (17.0.19+10-1~deb12u2).
0 

In [ ]:
# Definicion de los archivos que utilizamos
# Suponiendo que los archivos están en la misma carpeta que el notebook
path_jan = 'yellow_tripdata_2026-01.parquet'
path_feb = 'yellow_tripdata_2026-02.parquet'
path_mar = 'yellow_tripdata_2026-03.parquet'

# Leemos el archivo parquet y lo guardamos en un DataFrame
df_jan = spark.read.parquet(path_jan)
df_feb = spark.read.parquet(path_feb)
df_mar = spark.read.parquet(path_mar)

# Unimos los archivos de enero, febrero y marzo
df = df_jan.union(df_feb)
df = df.union(df_mar)

df.printSchema()

# Filas totales
print(f'Total de registros: {df.count()}')

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Total de registros: 11077206


# **Data Preprocessing**

En esta sección nuestro objetivo es detectar registros valores nulos o irreales en campos clave, una vez detectados para facilitar el análisis posterior los eliminaremos.

## Conteo Inicial del Total de Registros y de Valores Null

In [ ]:
# Conteo inicial para documentar (ver el total de datos antes de la limpieza)
total_inicial = df.count() #numero de filas del dataframe unificado
print(f"Total de registros iniciales: {total_inicial}")

# Calcular la cantidad de valores nulos por cada columna
nulos_por_columna = df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])

# Resultado
print("Cantidad de valores nulos por columna:")
nulos_por_columna.show(vertical=True)

Total de registros iniciales: 11077206
Cantidad de valores nulos por columna:
-RECORD 0------------------------
 VendorID              | 0       
 tpep_pickup_datetime  | 0       
 tpep_dropoff_datetime | 0       
 passenger_count       | 3057123 
 trip_distance         | 0       
 RatecodeID            | 3057123 
 store_and_fwd_flag    | 3057123 
 PULocationID          | 0       
 DOLocationID          | 0       
 payment_type          | 0       
 fare_amount           | 0       
 extra                 | 0       
 mta_tax               | 0       
 tip_amount            | 0       
 tolls_amount          | 0       
 improvement_surcharge | 0       
 total_amount          | 0       
 congestion_surcharge  | 3057123 
 Airport_fee           | 3057123 
 cbd_congestion_fee    | 0       



Al revisar los más de 11 millones de registros iniciales, detectamos que cinco columnas (cantidad de pasajeros, código de tarifa, recargos por congestión, tarifa de aeropuerto y el flag de almacenamiento) presentan exactamente 3.057.123 valores nulos cada una. Esta cifra idéntica apunta a un fallo técnico sistemático de algún proveedor al registrar los datos, no a una pérdida aleatoria. A pesar de esto, el análisis es totalmente viable porque las columnas realmente críticas para medir la rentabilidad y la movilidad, como la distancia del trayecto y el importe final pagado, están completas y no tienen ningún valor nulo.

## Eliminamos Registros con Valores Nulos en Columnas Clave

In [ ]:
# Seleccionar solo las columnas que nos interesan para el análisis
columnas_importantes = ["passenger_count", "trip_distance", "fare_amount"]
df_limpio = df.dropna(subset=columnas_importantes) #eliminar filas con valores nulos

# Conteo final para ver cuántos registros hemos perdido
total_sin_nulos = df_limpio.count()

# Calcular la cantidad de registros eliminados
registros_eliminados = total_inicial - total_sin_nulos

# Resultados
print(f"Total de registros tras limpiar nulos: {total_sin_nulos}")
print(f"Registros eliminados por falta de datos: {registros_eliminados} ({(registros_eliminados/total_inicial)*100:.2f}%)")

Total de registros tras limpiar nulos: 8020083
Registros eliminados por falta de datos: 3057123 (27.60%)


## Eliminamos Registros con Valores Irreales o Fisicamente Imposibles ##

In [ ]:
# Guardamos el conteo de antes de aplicar los filtros
total_antes_filtros = df_limpio.count()

# Filtro de distancia: un viaje debe tener una distancia mayor a 0
df_limpio = df_limpio.filter(col("trip_distance") > 0)

# Filtro de tarifas: eliminamos tarifas en cero o negativas (suelen ser cancelaciones o errores)
df_limpio = df_limpio.filter((col("fare_amount") > 0) & (col("total_amount") > 0))

# Filtro de tiempo (duración): la fecha/hora de bajada debe ser estrictamente mayor a la de subida.
df_limpio = df_limpio.filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))

# Conteo posterior y cálculo de eliminados
total_despues_filtros = df_limpio.count()
registros_filtrados = total_antes_filtros - total_despues_filtros

# Resultados
print(f"Registros eliminados por ceros/negativos: {registros_filtrados}")
print(f"Total de registros válidos hasta ahora: {total_despues_filtros}")

Registros eliminados por ceros/negativos: 312787
Total de registros válidos hasta ahora: 7707296


## Eliminamos Registros con Cantidad de Pasajeros Irreal

In [ ]:
# Guardamos el conteo antes de aplicar los filtros
total_antes_pasajeros = df_limpio.count()

# Filtramos viajes con 0 pasajeros o con más de 6 (límite realista)
df_limpio = df_limpio.filter((col("passenger_count") > 0) & (col("passenger_count") <= 6))

# Calculo de los eliminados
registros_eliminados_pasajeros = total_antes_pasajeros - df_limpio.count()

# Resultados
print(f"Registros eliminados por cantidad irreal de pasajeros: {registros_eliminados_pasajeros}")

Registros eliminados por cantidad irreal de pasajeros: 38652


## Eliminamos Registros con Fechas Fuera del Rango de Estudio

In [ ]:
# Definimos año de estudio
año_analisis = 2026

# Guardamlos las filas antes de aplicar los filtros
total_antes_tiempo = df_limpio.count()

# Validar que el año de recogida sea el correcto
df_limpio = df_limpio.filter(year(col("tpep_pickup_datetime")) == año_analisis)

# Validar que el año de bajada
# Permitimos que la bajada sea en el mismo año, o a lo sumo en los primeros días del año siguiente
df_limpio = df_limpio.filter(
    (year(col("tpep_dropoff_datetime")) == año_analisis) |
    (year(col("tpep_dropoff_datetime")) == año_analisis + 1)
)

# Calculo de los eliminados
registros_eliminados_tiempo = total_antes_tiempo - df_limpio.count()

# Resultados
print(f"Registros eliminados por años fuera de rango temporal: {registros_eliminados_tiempo}")

# Guardamos el conteo final después de la limpieza
total_final_limpio = df_limpio.count()

Registros eliminados por años fuera de rango temporal: 7


In [ ]:
# Guardamos el total antes del filtro temporal estricto
total_antes_meses = df_limpio.count()

# Definimos los meses válidos para nuestro estudio (1=Enero, 2=Febrero, 3=Marzo)
meses_validos = [1, 2, 3]

# Filtramos usando la función isin()
df_limpio = df_limpio.filter(month(col("tpep_pickup_datetime")).isin(meses_validos))

# Calculo de los eliminados
registros_eliminados_meses = total_antes_meses - df_limpio.count()

# Resultados
print(f"Registros fuera del trimestre Ene-Mar eliminados: {registros_eliminados_meses}")


# Resultados finales
total_final_limpio = df_limpio.count()
total_eliminados_global = total_inicial - total_final_limpio

print("       --- LIMPIEZA COMPLETADA ---")
print(f"Registros iniciales:      {total_inicial}")
print(f"Registros eliminados:     {total_eliminados_global}")
print(f"Registros válidos finales: {total_final_limpio}")
print(f"Efectividad de los datos:  {(total_final_limpio/total_inicial)*100:.2f}%")


Registros fuera del trimestre Ene-Mar eliminados: 2
       --- LIMPIEZA COMPLETADA ---
Registros iniciales:      11077206
Registros eliminados:     3408571
Registros válidos finales: 7668635
Efectividad de los datos:  69.23%
